# 12. Query, Key und ValueAus *Transformer von Grund auf*.

In [ ]:
# In Colab liegt nichts auf der Festplatte, also holt diese Zelle# die zwei oder drei Dateien, die das Kapitel braucht. Wer das# Repository geklont hat, findet sie schon vor, und es wird# nichts heruntergeladen.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/notebooks/de/code/"PLACES = [".", "code", "../code"]for name in ["ch12_qkv.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Query, Key und Value: ein Nachschlagen, das mischen darf.Ein gewöhnliches Wörterbuch findet einen Key genau oder scheitert. Die Attention bewertet jeden Key, macht aus den Scores Anteile und gibt eine Mischung zurück. Diese Datei stellt beides nebeneinander.

In [ ]:
import numpy as npnp.set_printoptions(precision=3, suppress=True, floatmode="fixed")KEYS = np.array([[1.0, 0.0],                 [0.0, 1.0],                 [-1.0, 0.0],                 [0.0, -1.0]])VALUES = np.array([[10.0, 0.0],                   [0.0, 10.0],                   [-10.0, 0.0],                   [0.0, -10.0]])NAMES = ["Nord", "Ost", "Süd", "West"]

In [ ]:
def softmax(scores):    shifted = scores - scores.max()    e = np.exp(shifted)    return e / e.sum()

Ein Wörterbuch. Ein Key passt genau, oder es kommt nichts.

In [ ]:
def exact_lookup(query):    for name, key, value in zip(NAMES, KEYS, VALUES):        if np.allclose(query, key):            return name, value    return None, None

Jeden Key bewerten, normieren, dann die Values danach mischen.

In [ ]:
def soft_lookup(query, sharpness=4.0):    scores = KEYS @ query * sharpness    weights = softmax(scores)    return weights, weights @ VALUES

Aus einer Eingabezeile werden drei verschiedene Zeilen.

In [ ]:
def projections(x, w_q, w_k, w_v):    return x @ w_q, x @ w_k, x @ w_v

In [ ]:
def main():    print("EIN GEWÖHNLICHES NACHSCHLAGEN")    for query in [np.array([1.0, 0.0]), np.array([0.7, 0.7])]:        name, value = exact_lookup(query)        found = f"{name}, {value}" if name else "überhaupt nichts"        print(f"  Query {query}  ->  {found}")    print("\nDIESELBEN ZWEI QUERIES, BEWERTET UND GEMISCHT")    for query in [np.array([1.0, 0.0]), np.array([0.7, 0.7])]:        weights, blended = soft_lookup(query)        print(f"  Query {query}")        for name, weight in zip(NAMES, weights):            print(f"    {name:<7} {weight:.3f}")        print(f"    Ergebnis {blended}")    print("\nEINE QUERY, DIE NICHTS TRIFFT, BEKOMMT TROTZDEM ANTWORT")    weights, blended = soft_lookup(np.array([0.0, 0.0]))    print("  Query [0. 0.]")    print("    Gewichte", weights)    print("    Ergebnis", blended)    print("  jeder Key bekam null, also mischt die Antwort gleichmäßig")    print("\nEINE EINGABE, DREI VERSCHIEDENE VEKTOREN")    x = np.array([1.0, 2.0])    w_q = np.array([[1.0, 0.0], [0.0, 1.0]])    w_k = np.array([[0.0, 1.0], [1.0, 0.0]])    w_v = np.array([[2.0, 0.0], [0.0, 0.5]])    q, k, v = projections(x, w_q, w_k, w_v)    print("  Eingabe ", x)    print("  Query   ", q, " wonach diese Position sucht")    print("  Key     ", k, " was sie zum Abgleich anbietet")    print("  Value   ", v, " was sie beiträgt, wenn man auf sie achtet")    print("  drei Rollen, drei Matrizen, eine Eingabe")

In [ ]:
main()

Das Kapitel, aus dem dieses Notebook stammt, rechnet jede Zahl oben durch und erklärt, warum jeder Schritt da ist.